In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, time, warnings
os.chdir('/content/drive/MyDrive/network-intrusion-ml')
sys.path.append('/content/drive/MyDrive/network-intrusion-ml')

import numpy as np
import pandas as pd

from src.utils import (PROCESSED_DIR, METRIC_DIR, TABLE_DIR, save_table,
                       log_experiment, RANDOM_STATE)

train_df = pd.read_parquet(PROCESSED_DIR / "train.parquet")   # NOT test.parquet

with open(METRIC_DIR / "label_mapping.json") as f:
    label_mapping = json.load(f)
with open(METRIC_DIR / "feature_sets.json") as f:
    feature_sets = json.load(f)

FEATURES = feature_sets["CORR_FILTERED"]
class_names = [c for c, i in sorted(label_mapping.items(), key=lambda kv: kv[1])]
print("Train rows:", f"{len(train_df):,}", "| Features used:", len(FEATURES))

Mounted at /content/drive
Train rows: 1,998,384 | Features used: 48


In [2]:
%%writefile src/preprocessing.py
import numpy as np


def signed_log1p(X):
    """sign(x) * log(1 + |x|): shrinks huge values but keeps the sign."""
    return np.sign(X) * np.log1p(np.abs(X))

Writing src/preprocessing.py


In [3]:
import importlib
importlib.invalidate_caches()

def stratified_sample(df, n_total, min_per_class, seed):
    """Random sample that keeps class proportions, but never fewer than min_per_class rows per class."""
    frac = n_total / len(df)
    parts = []
    for cls, g in df.groupby("Label_Group"):
        n = min(len(g), max(int(round(len(g) * frac)), min_per_class))
        parts.append(g.sample(n=n, random_state=seed))
    return pd.concat(parts).sample(frac=1, random_state=seed)

dev_train  = stratified_sample(train_df, 300_000, 6, RANDOM_STATE)   # for LR, DT, RF, XGB
rest       = train_df.drop(index=dev_train.index)                    # rows not used for training
dev_val    = stratified_sample(rest, 100_000, 2, RANDOM_STATE)       # validation (no overlap)
slow_train = stratified_sample(dev_train, 50_000, 6, RANDOM_STATE)   # for KNN and SVM

sizes = pd.DataFrame({
    "dev_train": dev_train["Label_Group"].value_counts(),
    "slow_train (KNN, SVM)": slow_train["Label_Group"].value_counts(),
    "dev_val": dev_val["Label_Group"].value_counts(),
}).fillna(0).astype(int)
print(sizes)
print("\nTotals:", len(dev_train), len(slow_train), len(dev_val))
save_table(sizes, "04_dev_sample_sizes")

# Save which rows were used, so later notebooks can reuse the same samples
np.save(PROCESSED_DIR / "dev_train_idx.npy", dev_train.index.values)
np.save(PROCESSED_DIR / "dev_val_idx.npy", dev_val.index.values)

# Inputs and targets
X_tr, X_slow, X_val = dev_train[FEATURES], slow_train[FEATURES], dev_val[FEATURES]
def to_targets(d):
    return {"binary": d["Binary"].values,
            "multiclass": d["Label_Group"].map(label_mapping).values}
y = {"train": to_targets(dev_train), "slow": to_targets(slow_train), "val": to_targets(dev_val)}

del train_df, rest   # free memory

              dev_train  slow_train (KNN, SVM)  dev_val
Label_Group                                            
BENIGN           248872                  41478    82957
Bot                 234                     39       78
DDoS              15374                   2562     5125
DoS               23266                   3878     7756
FTP-Patator         712                    119      237
Heartbleed            6                      6        2
Infiltration          6                      6        2
PortScan          10892                   1815     3631
SSH-Patator         387                     64      129
Web Attack          257                     43       86

Totals: 300006 50010 100003
[saved table] /content/drive/MyDrive/network-intrusion-ml/results/tables/04_dev_sample_sizes.csv


In [4]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score,
                             precision_recall_fscore_support)

from src.preprocessing import signed_log1p

SCALED_MODELS = ("Logistic Regression", "KNN", "SVM (RBF)")

def scaled(model):
    """Signed log -> standard scaling -> model, all in one pipeline."""
    return Pipeline([("signed_log", FunctionTransformer(signed_log1p)),
                     ("scale", StandardScaler()),
                     ("model", model)])

def make_models():
    return {
        "Logistic Regression": (scaled(LogisticRegression(max_iter=300)), "full"),
        "KNN": (scaled(KNeighborsClassifier(n_neighbors=5, n_jobs=-1)), "slow"),
        "Decision Tree": (DecisionTreeClassifier(random_state=RANDOM_STATE), "full"),
        "Random Forest": (RandomForestClassifier(n_estimators=100, n_jobs=-1,
                                                 random_state=RANDOM_STATE), "full"),
        "SVM (RBF)": (scaled(SVC(kernel="rbf", C=1.0, gamma="scale")), "slow"),
        "XGBoost": (XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.3,
                                  tree_method="hist", n_jobs=-1,
                                  random_state=RANDOM_STATE, verbosity=0), "full"),
    }

def evaluate(task, model, X_v, y_v):
    t0 = time.time()
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X_v)
        pred = model.classes_[proba.argmax(axis=1)]
        score = proba[:, 1] if task == "binary" else None
    elif task == "binary":
        score = model.decision_function(X_v)
        pred = (score > 0).astype(int)
    else:
        pred, score = model.predict(X_v), None
    pred_s = time.time() - t0

    if task == "binary":
        m = {"accuracy": accuracy_score(y_v, pred),
             "precision": precision_score(y_v, pred, zero_division=0),
             "recall": recall_score(y_v, pred, zero_division=0),
             "f1": f1_score(y_v, pred, zero_division=0),
             "roc_auc": roc_auc_score(y_v, score),
             "pr_auc": average_precision_score(y_v, score)}
    else:
        present = np.unique(y_v)
        m = {"accuracy": accuracy_score(y_v, pred),
             "macro_precision": precision_score(y_v, pred, labels=present, average="macro", zero_division=0),
             "macro_recall": recall_score(y_v, pred, labels=present, average="macro", zero_division=0),
             "macro_f1": f1_score(y_v, pred, labels=present, average="macro", zero_division=0),
             "weighted_f1": f1_score(y_v, pred, average="weighted", zero_division=0)}
        p, r, f, s = precision_recall_fscore_support(
            y_v, pred, labels=list(range(len(class_names))), zero_division=0)
        for i, name in enumerate(class_names):
            m[f"f1_{name}"] = np.nan if s[i] == 0 else f[i]
    return m, pred_s

def run_task(task, csv_name):
    """Train every model, score on the validation slice, SAVE after each model.
    If Colab disconnects, run this again: finished models are skipped."""
    out = TABLE_DIR / csv_name
    rows = pd.read_csv(out).to_dict("records") if out.exists() else []
    done = {r["model"] for r in rows}

    for name, (model, size) in make_models().items():
        if name in done:
            print(f"[skip] {name} (already saved)")
            continue
        which = "slow" if size == "slow" else "train"
        X_fit = X_slow if size == "slow" else X_tr
        y_fit = y[which][task]

        t0 = time.time()
        model.fit(X_fit, y_fit)
        fit_s = time.time() - t0

        m, pred_s = evaluate(task, model, X_val, y["val"][task])
        row = {"model": name, "train_rows": len(X_fit), "train_seconds": round(fit_s, 1),
               "predict_seconds": round(pred_s, 1), **m}
        rows.append(row)
        pd.DataFrame(rows).to_csv(out, index=False)          # saved after every model

        log_experiment(
            experiment_id=f"EXP-02_{task}_{name}".replace(" ", "_"),
            task="Experiment A: binary" if task == "binary" else "Experiment B: multi-class (Label_Group)",
            model=name, features=f"CORR_FILTERED ({len(FEATURES)})",
            preprocessing="signed log + standard scaling" if name in SCALED_MODELS else "raw features",
            hyperparameters=f"baseline defaults; trained on {len(X_fit):,} rows",
            split="validation slice taken from the training set (test set untouched)",
            metrics={k: round(float(v), 4) for k, v in m.items() if not k.startswith("f1_")},
            notes=f"fit {fit_s:.0f}s, predict {pred_s:.0f}s",
        )
        key = "f1" if task == "binary" else "macro_f1"
        print(f"{name:20s} fit {fit_s:7.1f}s | accuracy {m['accuracy']:.4f} | {key} {m[key]:.4f}")
    return pd.DataFrame(rows)

print("Helpers ready.")

Helpers ready.


In [5]:
results_binary = run_task("binary", "04_baseline_binary.csv")

[logged experiment] EXP-02_binary_Logistic_Regression
Logistic Regression  fit    19.2s | accuracy 0.9797 | f1 0.9400
[logged experiment] EXP-02_binary_KNN
KNN                  fit     0.1s | accuracy 0.9967 | f1 0.9904
[logged experiment] EXP-02_binary_Decision_Tree
Decision Tree        fit    20.4s | accuracy 0.9979 | f1 0.9939
[logged experiment] EXP-02_binary_Random_Forest
Random Forest        fit    97.8s | accuracy 0.9984 | f1 0.9952
[logged experiment] EXP-02_binary_SVM_(RBF)
SVM (RBF)            fit     8.3s | accuracy 0.9956 | f1 0.9870
[logged experiment] EXP-02_binary_XGBoost
XGBoost              fit     7.6s | accuracy 0.9988 | f1 0.9966


In [6]:
results_multi = run_task("multiclass", "04_baseline_multiclass.csv")

[logged experiment] EXP-02_multiclass_Logistic_Regression
Logistic Regression  fit    29.7s | accuracy 0.9952 | macro_f1 0.8335
[logged experiment] EXP-02_multiclass_KNN
KNN                  fit     0.1s | accuracy 0.9966 | macro_f1 0.8143
[logged experiment] EXP-02_multiclass_Decision_Tree
Decision Tree        fit    22.4s | accuracy 0.9981 | macro_f1 0.8729
[logged experiment] EXP-02_multiclass_Random_Forest
Random Forest        fit   107.4s | accuracy 0.9983 | macro_f1 0.8719
[logged experiment] EXP-02_multiclass_SVM_(RBF)
SVM (RBF)            fit     5.8s | accuracy 0.9954 | macro_f1 0.6936
[logged experiment] EXP-02_multiclass_XGBoost
XGBoost              fit    68.4s | accuracy 0.9986 | macro_f1 0.9221


In [7]:
cols_b = ["model", "train_rows", "accuracy", "precision", "recall", "f1",
          "roc_auc", "pr_auc", "train_seconds", "predict_seconds"]
print("EXPERIMENT A: binary\n")
print(pd.read_csv(TABLE_DIR / "04_baseline_binary.csv")[cols_b].round(4).to_string(index=False))

cols_m = ["model", "train_rows", "accuracy", "macro_precision", "macro_recall",
          "macro_f1", "weighted_f1", "train_seconds"]
print("\nEXPERIMENT B: multi-class\n")
print(pd.read_csv(TABLE_DIR / "04_baseline_multiclass.csv")[cols_m].round(4).to_string(index=False))

EXPERIMENT A: binary

              model  train_rows  accuracy  precision  recall     f1  roc_auc  pr_auc  train_seconds  predict_seconds
Logistic Regression      300006    0.9797     0.9474  0.9327 0.9400   0.9934  0.9729           19.2              0.2
                KNN       50010    0.9967     0.9901  0.9907 0.9904   0.9979  0.9926            0.1             47.0
      Decision Tree      300006    0.9979     0.9929  0.9950 0.9939   0.9969  0.9890           20.4              0.0
      Random Forest      300006    0.9984     0.9959  0.9945 0.9952   0.9996  0.9970           97.8              0.7
          SVM (RBF)       50010    0.9956     0.9926  0.9815 0.9870   0.9977  0.9962            8.3             16.2
            XGBoost      300006    0.9988     0.9955  0.9977 0.9966   1.0000  0.9998            7.6              0.3

EXPERIMENT B: multi-class

              model  train_rows  accuracy  macro_precision  macro_recall  macro_f1  weighted_f1  train_seconds
Logistic Regression 